In [1]:
%pwd

'a:\\projects\\text-summarizer\\research'

In [2]:
import os
os.chdir('../')

In [3]:
%pwd

'a:\\projects\\text-summarizer'

# Entity

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: str
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    eval_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int

# Configuration manager

In [5]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories
from pathlib import Path

In [6]:
class ConfigurationManager:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories(path_to_directories=[Path(self.config.artifacts_root)])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments
        root_dir = Path(config.root_dir)
        
        create_directories(path_to_directories=[root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            model_ckpt=config.model_ckpt,
            num_train_epochs=params.num_train_epochs,
            warmup_steps=params.warmup_steps,
            per_device_train_batch_size=params.per_device_train_batch_size,
            weight_decay=params.weight_decay,
            logging_steps=params.logging_steps,
            eval_strategy=params.eval_strategy,
            eval_steps=params.eval_steps,
            save_steps=int(float(params.save_steps)),
            gradient_accumulation_steps=params.gradient_accumulation_steps
        )

        return model_trainer_config

# Componenets

In [7]:
import os
import torch
from datasets import load_from_disk
from transformers import (
    TrainingArguments, 
    Trainer,
    DataCollatorForSeq2Seq,
    AutoModelForSeq2SeqLM, 
    AutoTokenizer
)
from textSummarizer.logging import logger

a:\projects\env\text_summarizer\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        logger.info(f"Model training initiated. Using device: {device}")

        tokenizer_path = os.path.join(os.path.dirname(self.config.model_ckpt), "tokenizer")
        logger.info(f"Loading local tokenizer from {tokenizer_path}")
        
        tokenizer = AutoTokenizer.from_pretrained(
            tokenizer_path,
            use_fast=False
        )
        logger.info("Tokenizer loaded successfully. Loading optimized model weights...")
        
        # Low memory optimization configuration
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_ckpt,
            low_cpu_mem_usage=True,        
            torch_dtype=torch.float16 if device == "cuda" else torch.float32  
        ).to(device)
        
        logger.info(f"Model weights successfully instantiated and loaded onto {device}!")
        
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)
        
        # Load tokenized arrow datasets from disk
        logger.info(f"Loading tokenized datasets from path: {self.config.data_path}")
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        # Fully dynamic Training Arguments using dataclass config
        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir, 
            num_train_epochs=self.config.num_train_epochs, 
            warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size, 
            per_device_eval_batch_size=self.config.per_device_train_batch_size,
            weight_decay=self.config.weight_decay, 
            logging_steps=self.config.logging_steps,
            eval_strategy=self.config.eval_strategy, 
            eval_steps=self.config.eval_steps, 
            save_steps=int(float(self.config.save_steps)),
            gradient_accumulation_steps=self.config.gradient_accumulation_steps
        ) 

        trainer = Trainer(
            model=model_pegasus, 
            args=trainer_args,
            processing_class=tokenizer, 
            data_collator=seq2seq_data_collator,
            
            train_dataset=dataset_samsum_pt["train"], 
            eval_dataset=dataset_samsum_pt["validation"]
        )
        
        logger.info("Starting Trainer pipeline execution...")
        trainer.train()
        logger.info("Model training completed successfully!")

        # Save model artifacts
        logger.info(f"Saving final model and tokenizer artifacts to: {self.config.root_dir}")
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir, "pegasus-samsum-model"))
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))
        logger.info("Artifacts saved successfully. Pipeline stage complete.")

# Pipeline

In [16]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    raise e

[2026-06-25 09:44:38,226: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-25 09:44:38,233: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-25 09:44:38,234: INFO: common: Created directory at: artifacts]
[2026-06-25 09:44:38,238: INFO: common: Created directory at: artifacts\model_trainer]
[2026-06-25 09:44:38,240: INFO: 3556790641: Model training initiated. Using device: cpu]
[2026-06-25 09:44:38,240: INFO: 3556790641: Loading local tokenizer from artifacts/model_trainer\tokenizer]
[2026-06-25 09:44:39,877: INFO: 3556790641: Tokenizer loaded successfully. Loading optimized model weights...]


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 680/680 [00:00<00:00, 2816.79it/s]


[2026-06-25 09:44:44,635: INFO: 3556790641: Model weights successfully instantiated and loaded onto cpu!]
[2026-06-25 09:44:44,637: INFO: 3556790641: Loading tokenized datasets from path: artifacts\data_transformation\samsum_dataset]
[2026-06-25 09:44:44,913: INFO: 3556790641: Starting Trainer pipeline execution...]


a:\projects\env\text_summarizer\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,1.310485
2,2.870387
3,1.850101
4,1.601346
5,1.564750
6,1.004990
7,2.107316
8,2.347430
9,1.147409
10,1.409658


KeyboardInterrupt: 